# Create MCP Connection

Creates or updates a **schema-level** UC HTTP connection for the MCP service.
Schema-level connections need `CREATE CONNECTION` on the schema (covered by
`ALL PRIVILEGES` on the catalog) — no metastore grant needed.

Run by `deploy.sh` Phase 2 via `databricks bundle run create_mcp_connection`.

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("connection_name", "", "Connection name")
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema_name", "", "Schema")
dbutils.widgets.text("app_url", "", "App URL")
dbutils.widgets.text("base_path", "/mcp", "Base path")
dbutils.widgets.text("scope_name", "", "Secret scope")
dbutils.widgets.text("client_id_key", "", "Client ID key")
dbutils.widgets.text("client_secret_key", "", "Client secret key")
dbutils.widgets.text("oauth_scopes", "apps ai-gateway", "OAuth scopes")
dbutils.widgets.text("token_endpoint", "", "Token endpoint")

connection_name = dbutils.widgets.get("connection_name")
catalog = dbutils.widgets.get("catalog")
schema_name = dbutils.widgets.get("schema_name")
app_url = dbutils.widgets.get("app_url")
base_path = dbutils.widgets.get("base_path")
scope_name = dbutils.widgets.get("scope_name")
client_id_key = dbutils.widgets.get("client_id_key")
client_secret_key = dbutils.widgets.get("client_secret_key")
oauth_scopes = dbutils.widgets.get("oauth_scopes")
token_endpoint = dbutils.widgets.get("token_endpoint")

full_name = f"{catalog}.{schema_name}.{connection_name}"
parent = f"schemas/{catalog}.{schema_name}"

print(f"Connection: {full_name}")
print(f"Parent:     {parent}")
print(f"App URL:    {app_url}")

In [0]:
client_id = dbutils.secrets.get(scope=scope_name, key=client_id_key)
client_secret = dbutils.secrets.get(scope=scope_name, key=client_secret_key)

options = {
    "host": app_url,
    "port": "443",
    "base_path": base_path,
    "client_id": client_id,
    "client_secret": client_secret,
    "oauth_scope": oauth_scopes,
    "token_endpoint": token_endpoint,
}

print(f"Read credentials from scope={scope_name}")
print(f"  client_id_key={client_id_key}")
print(f"  client_secret_key={client_secret_key}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

try:
    existing = w.api_client.do("GET", f"/api/2.1/unity-catalog/connections/{full_name}")
    print(f"Connection exists: {full_name} \u2014 updating options")
    result = w.api_client.do(
        "PATCH",
        f"/api/2.1/unity-catalog/connections/{full_name}",
        body={"options": options},
    )
    print(f"Updated: {result.get('full_name', full_name)}")
except Exception as e:
    err = str(e)
    if "does not exist" in err or "NOT_FOUND" in err:
        print(f"Connection does not exist \u2014 creating: {full_name}")
        result = w.api_client.do(
            "POST",
            "/api/2.1/unity-catalog/connections",
            body={
                "name": connection_name,
                "parent": parent,
                "connection_type": "HTTP",
                "options": options,
            },
        )
        print(f"Created: {result.get('full_name', full_name)}")
    else:
        raise

In [0]:
import json

verify = w.api_client.do("GET", f"/api/2.1/unity-catalog/connections/{full_name}")
print(f"Verified: {verify.get('full_name')}")
print(f"  credential_type: {verify.get('credential_type')}")
print(f"  url: {verify.get('url')}")

# Redact secrets from options before displaying
safe_opts = {k: v for k, v in verify.get("options", {}).items() if k != "client_secret"}
print(f"  options: {json.dumps(safe_opts, indent=2)}")